In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import timm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split
)

from torchvision import transforms
from PIL import Image

# =========================================
# CONFIG
# =========================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

REAL_PATH = "dataset/real"
FAKE_PATH = "dataset/fake"

IMG_SIZE = 224
NUM_FRAMES = 8

BATCH_SIZE = 4
EPOCHS = 20

LEARNING_RATE = 0.00005

# =========================================
# OPENCV FACE DETECTION
# =========================================

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    'haarcascade_frontalface_default.xml'
)

# =========================================
# FRAME EXTRACTION
# =========================================

def extract_frames(video_path, num_frames=8):

    cap = cv2.VideoCapture(video_path)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    frames = []

    if total_frames == 0:
        return frames

    step = max(total_frames // num_frames, 1)

    for i in range(num_frames):

        frame_id = i * step

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_id
        )

        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(frame)

    cap.release()

    return frames

# =========================================
# FACE CROPPING
# =========================================

def crop_face(frame):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_RGB2GRAY
    )

    faces = face_cascade.detectMultiScale(

        gray,

        scaleFactor=1.1,

        minNeighbors=5,

        minSize=(60, 60)
    )

    if len(faces) > 0:

        x, y, w, h = faces[0]

        face = frame[y:y+h, x:x+w]

        if face.size != 0:
            return face

    return frame

# =========================================
# TRANSFORM
# =========================================

transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# =========================================
# DATASET
# =========================================

class DeepfakeDataset(Dataset):

    def __init__(self, real_path, fake_path):

        self.samples = []

        # REAL
        for file in os.listdir(real_path):

            if file.endswith(".mp4"):

                self.samples.append(
                    (
                        os.path.join(real_path, file),
                        0
                    )
                )

        # FAKE
        for file in os.listdir(fake_path):

            if file.endswith(".mp4"):

                self.samples.append(
                    (
                        os.path.join(fake_path, file),
                        1
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        video_path, label = self.samples[idx]

        frames = extract_frames(
            video_path,
            NUM_FRAMES
        )

        processed_frames = []

        for frame in frames:

            # FACE CROP
            face = crop_face(frame)

            image = Image.fromarray(face)

            image = transform(image)

            processed_frames.append(image)

        # Padding if frames missing
        while len(processed_frames) < NUM_FRAMES:

            processed_frames.append(
                torch.zeros(
                    3,
                    IMG_SIZE,
                    IMG_SIZE
                )
            )

        frames_tensor = torch.stack(
            processed_frames
        )

        return (
            frames_tensor,
            torch.tensor(
                label,
                dtype=torch.float32
            )
        )

# =========================================
# MODEL
# =========================================

class DeepfakeModel(nn.Module):

    def __init__(self):

        super(DeepfakeModel, self).__init__()

        # XCEPTION
        self.xception = timm.create_model(
            'xception',
            pretrained=True,
            num_classes=0
        )

        # Freeze Xception initially
        #for param in self.xception.parameters():
           # param.requires_grad = False

        # CLASSIFIER
        self.classifier = nn.Sequential(

            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 1)
        )

    def forward(self, x):

        # Shape:
        # [batch, frames, channels, H, W]

        batch_size, num_frames, C, H, W = x.shape

        frame_features = []

        for i in range(num_frames):

            frame = x[:, i]

            features = self.xception(frame)

            frame_features.append(features)

        frame_features = torch.stack(
            frame_features,
            dim=1
        )

        # MAX pooling
        video_features = torch.max(
            frame_features,
            dim=1
        )[0]

        output = self.classifier(
            video_features
        )

        return output.squeeze()

# =========================================
# LOAD DATASET
# =========================================

dataset = DeepfakeDataset(
    REAL_PATH,
    FAKE_PATH
)

print("Total Samples:", len(dataset))

real_count = 0
fake_count = 0

for _, label in dataset.samples:

    if label == 0:
        real_count += 1
    else:
        fake_count += 1

print("Real Videos:", real_count)
print("Fake Videos:", fake_count)

# =========================================
# TRAIN / VALIDATION SPLIT
# =========================================

train_size = int(0.8 * len(dataset))

val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

# =========================================
# DATALOADERS
# =========================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=True
)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=True
)

# =========================================
# MODEL SETUP
# =========================================

model = DeepfakeModel().to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# =========================================
# TRAINING LOOP
# =========================================

for epoch in range(EPOCHS):

    # =====================================
    # TRAIN
    # =====================================

    model.train()

    train_loss = 0

    train_correct = 0

    train_total = 0

    loop = tqdm(
        train_loader,
        desc=f"Epoch [{epoch+1}/{EPOCHS}]"
    )

    for videos, labels in loop:

        videos = videos.to(DEVICE)

        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        predictions = (
            torch.sigmoid(outputs) > 0.5
        ).float()

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

        train_accuracy = (
            100 * train_correct / train_total
        )

        loop.set_postfix(
            loss=loss.item(),
            accuracy=train_accuracy
        )

    # =====================================
    # VALIDATION
    # =====================================

    model.eval()

    val_loss = 0

    val_correct = 0

    val_total = 0

    with torch.no_grad():

        for videos, labels in val_loader:

            videos = videos.to(DEVICE)

            labels = labels.to(DEVICE)

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            predictions = (
                torch.sigmoid(outputs) > 0.5
            ).float()

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = (
        100 * val_correct / val_total
    )

    # =====================================
    # RESULTS
    # =====================================

    print("\n==============================")

    print(f"Epoch {epoch+1}/{EPOCHS}")

    print(
        f"Train Loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy: "
        f"{train_accuracy:.2f}%"
    )

    print(
        f"Validation Loss: "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: "
        f"{val_accuracy:.2f}%"
    )

    print("==============================")

# =========================================
# SAVE MODEL
# =========================================

torch.save(
    model.state_dict(),
    "deepfake_xception_face_model.pth"
)

print("\nModel Saved Successfully!")

i:\Research\Deepfake Detection program\deepfake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total Samples: 2000
Real Videos: 1000
Fake Videos: 1000


i:\Research\Deepfake Detection program\deepfake\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
Epoch [1/20]:  21%|██▏       | 85/400 [02:22<09:39,  1.84s/it, accuracy=62.1, loss=0.568]